<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sl-edu-reform-voices/blob/main/Solution_2410215.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilingual NLP Analysis of Public YouTube Discussion on Sri Lankan School Education Reforms
**Student ID:** 2410215 | **Module:** CM4608 Natural Language Processing

This notebook collects public YouTube comments about the Sri Lankan school education reforms (English, Sinhala and Tamil) and analyses them with NLP: data collection and EDA (Task 1), preprocessing and tokenisation (Task 2), theme discovery (Task 3), classification (Task 4), structured extraction (Task 5), and evidence-grounded summaries with an ethics discussion (Task 6).

### 0.1 Reproducibility
A fixed random seed is used for every random step, and package versions are printed so the results can be reproduced.

## 1. Task 1 — Video discovery, comment collection and initial exploration

### 1.1 Environment set-up and API connection test

**Purpose:** Connect to the YouTube Data API v3 and confirm that the API key works before collecting any data.

**Approach:**
- Use google-api-python-client, Google’s official Python client for its APIs.
- Read saved API Key Secrets from colab
- The key is restricted in Google Cloud Console to the YouTube Data API v3 only. No OAuth or user login is used, because only public comments are collected
- Run a single test search.

1. Install the library

In [7]:
!pip install -q google-api-python-client

from google.colab import userdata
from googleapiclient.discovery import build
import sys, random, importlib.metadata as md
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
for pkg in ["numpy", "pandas", "google-api-python-client"]:
    print(f"{pkg}: {md.version(pkg)}")

Python: 3.13.18
numpy: 2.4.6
pandas: 2.2.3
google-api-python-client: 2.201.0


2. Read API Key

In [8]:
API_KEY = userdata.get("YOUTUBE_API_KEY")

3. Create the YouTube Client

In [9]:
yt = build("youtube", "v3", developerKey=API_KEY)

In [11]:
# Connection test: one search
# Ranking changes over time
r = yt.search().list(q="Sri Lanka education reform", part="snippet",
                     type="video", maxResults=3, regionCode="LK").execute()
for item in r["items"]:
    print(item["id"]["videoId"], "|", item["snippet"]["channelTitle"], "|", item["snippet"]["title"])

aUKv__hANms | Manoj Dhananjaya | New Education Reforms 2026 Sri Lanka | 2026 Education Reforms | GPA System for Schools | OL 2026
xzC_81piivo | Life with Tishi in SL | Major education reforms in Sri Lanka
76QErEQG89k | Ada Derana | New education reforms set for Grades 2 and 6 next year; Cabinet gives nod for 2027 implementation


QQGkW9hedpU | Betasoft | Grade 6 – New Education Reform 2026 – Subjects, Modules and Evaluation (in Tamil) |
| Mzt_IWl0mCE | Life with Tishi in SL | 1998 Education reforms and 2007 curriculum reforms in Sri Lanka |
| 1vF1RS7cMm0 | SCIENCE ATOM | அதிரடி மாற்றம்! இலங்கையில் 2026 முதல் புதிய கல்வி முறை – Sri Lanka Education Reforms 2026 |

**Output interpretation Oct 26, 2026:** The test returned three videos from three channels, confirming the key and client work. Two observations shaped the collection design: (1) even an English query returned Tamil-language videos, so the corpus will be multilingual; (2) one result concerned the 1998/2007 reforms, so searches are limited by publication date and videos are screened for relevance.

4.Storage


In [12]:
from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/NLP_sl-edu-reform-voices"
RAW_DIR  = f"{PROJECT_DIR}/data/raw"        # one file per video (raw API data)
PROC_DIR = f"{PROJECT_DIR}/data/processed"  # combined datasets
DOCS_DIR = f"{PROJECT_DIR}/docs"            # search log + video lists (reproducibility)

for d in (RAW_DIR, PROC_DIR, DOCS_DIR):
    os.makedirs(d, exist_ok=True)           # create folders if missing

print("Saving data under:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving data under: /content/drive/MyDrive/NLP_sl-edu-reform-voices


### 1.2 Collection configuration and logging

**Purpose:** Record the metadata required by Task 1(a) for reproducibility: every search query, the exact date and time it was run, the number of results, and the API quota used. All timestamps are in UTC so they are unambiguous.

In [6]:
import datetime, csv, json, time
from googleapiclient.errors import HttpError

REGION = "LK"                               # rank results as seen from Sri Lanka
PUBLISHED_AFTER = "2024-01-01T00:00:00Z"    # only recent reform discussion
COLLECTION_DATE = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
LOG_FILE = f"{DOCS_DIR}/search_log.csv"
QUOTA_USED = 0                              # units used in this Colab session

def now_utc():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")

def log_call(call_type, label, n_results, units):
    """Write one row per API call: when, what, how many results, quota cost."""
    global QUOTA_USED
    QUOTA_USED += units
    new_file = not os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["timestamp_utc", "call_type", "query_or_id",
                        "n_results", "quota_units", "session_quota_total"])
        w.writerow([now_utc(), call_type, label, n_results, units, QUOTA_USED])

def api_call(request, call_type, label, units, retries=3):
    """Run an API request safely: log it, retry temporary errors, skip bad videos."""
    for attempt in range(retries):
        try:
            resp = request.execute()
            log_call(call_type, label, len(resp.get("items", [])), units)
            return resp
        except HttpError as e:
            try:
                reason = json.loads(e.content)["error"]["errors"][0]["reason"]
            except Exception:
                reason = str(e.resp.status)
            log_call(f"{call_type}_ERROR", f"{label} ({reason})", 0, units)
            if reason == "quotaExceeded":
                raise RuntimeError("Daily quota used up. Continue after ~1:30 pm Sri Lanka time.")
            if e.resp.status in (500, 503) and attempt < retries - 1:
                time.sleep(2 ** attempt)        # wait 1s, 2s... then retry
                continue
            print(f"   skipped {label}: {reason}")   # e.g. commentsDisabled
            return None

print("Collection date (UTC):", COLLECTION_DATE)

Collection date (UTC): 2026-11-15


**Output interpretation Nov 15, 2026:** All timestamps are UTC. `PUBLISHED_AFTER` limits the search to videos from 2024 onwards because the connection test showed that older reforms (1998/2007) also appear in results

### 1.3 Video discovery by keyword search
Videos are found automatically with `search.list` (up to 50 results per query, published after 2024-01-01, ranked for Sri Lanka).

**Query design:**
- **Three languages** (English, Sinhala, Tamil) so that the corpus reflects all three language communities.
- **Initial queries** covered the general topic ("education reform", "අධ්‍යාපන ප්‍රතිසංස්කරණ", "கல்வி சீர்திருத்தம்").
- **Refined queries:** after inspecting real reform videos, terms used by creators were added: "modules", "2027" (some videos refer to a 2027 rollout), "6 ශ්‍රේණිය" (Grade 6) and "1 සිට 13 දක්වා" (Grades 1–13).
- **Tamil and English queries include "Sri Lanka" / "இலங்கை"**, because generic Tamil queries return Indian (Tamil Nadu) education videos.
- **Politician names were not used** as queries, to avoid biasing the corpus towards political rather than educational discussion.

In [13]:
SEARCH_QUERIES = [
    # English
    "Sri Lanka education reform",
    "Sri Lanka education reform 2026",
    "Sri Lanka education reform 2027",
    "Sri Lanka education reform modules",
    "Sri Lanka new curriculum grade 6",
    # Sinhala
    "අධ්‍යාපන ප්‍රතිසංස්කරණ",                  # education reforms
    "නව අධ්‍යාපන ප්‍රතිසංස්කරණ 2026",           # new education reforms 2026
    "අධ්‍යාපන ප්‍රතිසංස්කරණ 2027",              # education reforms 2027
    "නව අධ්‍යාපන ප්‍රතිසංස්කරණ modules",        # new reforms modules
    "6 ශ්‍රේණිය මොඩියුල",                      # grade 6 modules
    "1 සිට 13 දක්වා අධ්‍යාපන ප්‍රතිසංස්කරණ",                # education reform from grade 1 to 13
    # Tamil
    "இலங்கை கல்வி சீர்திருத்தம்",               # Sri Lanka education reform
    "இலங்கை கல்வி சீர்திருத்தம் 2026",          # ... 2026
    "இலங்கை புதிய கல்வி சீர்திருத்தம் தரம் 6",   # Sri Lanka new reform grade 6
]

found = {}   # video_id -> first query that found it
for q in SEARCH_QUERIES:
    r = api_call(yt.search().list(q=q, part="snippet", type="video",
                                  regionCode=REGION, publishedAfter=PUBLISHED_AFTER,
                                  maxResults=50, order="relevance"),
                 "search", q, 100)
    if r is None:
        continue
    new = sum(it["id"]["videoId"] not in found for it in r["items"])
    for it in r["items"]:
        found.setdefault(it["id"]["videoId"], q)
    print(f"{len(r['items']):3} results ({new:2} new) | {q}")

print(len(found), "unique candidate videos | quota used:", QUOTA_USED)

 50 results (50 new) | Sri Lanka education reform
 50 results (28 new) | Sri Lanka education reform 2026
 50 results (28 new) | Sri Lanka education reform 2027
 50 results (18 new) | Sri Lanka education reform modules
 50 results (36 new) | Sri Lanka new curriculum grade 6
 50 results (36 new) | අධ්‍යාපන ප්‍රතිසංස්කරණ
 50 results (14 new) | නව අධ්‍යාපන ප්‍රතිසංස්කරණ 2026
 50 results (25 new) | අධ්‍යාපන ප්‍රතිසංස්කරණ 2027
 50 results ( 2 new) | නව අධ්‍යාපන ප්‍රතිසංස්කරණ modules
 50 results (36 new) | 6 ශ්‍රේණිය මොඩියුල
 50 results (41 new) | 1 සිට 13 දක්වා අධ්‍යාපන ප්‍රතිසංස්කරණ
 50 results (46 new) | இலங்கை கல்வி சீர்திருத்தம்
 50 results (12 new) | இலங்கை கல்வி சீர்திருத்தம் 2026
 24 results ( 5 new) | இலங்கை புதிய கல்வி சீர்திருத்தம் தரம் 6
377 unique candidate videos | quota used: 1400


**Output interpretation (15 Nov 2026):** The 14 queries returned 674 results, which reduced to **377 unique candidate videos** (1,400 quota units).

- **The general queries found the most new videos:** "Sri Lanka education reform" (50 new), "இலங்கை கல்வி சீர்திருத்தம்" (46) and "1 සිට 13 දක්වා අධ්‍යාපන ප්‍රතිසංස්කරණ" (41). Each language therefore reaches a different set of videos, which supports searching in all three languages.
- **The refined Sinhala terms were useful:** "6 ශ්‍රේණිය මොඩියුල" (36 new) and "1 සිට 13 දක්වා" (41 new) found many videos that the general queries missed, which justifies adding creators' own terms.
- **Some queries overlap heavily:** "නව අධ්‍යාපන ප්‍රතිසංස්කරණ modules" added only 2 new videos, and the 2026 variants added 12–14, showing diminishing returns from adding more similar queries.
- **There is less Tamil content:** the Tamil Grade 6 query returned only 24 results (5 new), suggesting less Tamil-language video coverage of the reform. Tamil may therefore be under-represented in the corpus.
- The 377 candidates still include off-topic videos, so they are screened in 1.4.

### 1.4 Video details and relevance screening
Details (channel, date, comment count) are fetched for every candidate video, then screened in two steps:
* **1.4a.Automatic:** keep videos whose title or description contains a reform keyword (English, Sinhala or Tamil) and that have at least 20 comments.
Videos with fewer than 20 comments were skipped because they add little data for the effort of screening them.
* **1.4b.Manual:** the remaining titles were read, and off-topic videos were
excluded with a recorded reason.

In [18]:
rows, ids = [], list(found)
for i in range(0, len(ids), 50):                      # 50 videos per call = 1 unit
    r = api_call(yt.videos().list(part="snippet,statistics", id=",".join(ids[i:i+50])),
                 "videos", f"details batch {i//50 + 1}", 1)
    for v in r["items"]:
        s, st = v["snippet"], v["statistics"]
        rows.append({"video_id": v["id"], "channel_id": s["channelId"],
                     "channel_title": s["channelTitle"], "title": s["title"],
                     "published_at": s["publishedAt"],
                     "comment_count": int(st.get("commentCount", 0)),
                     "text": (s["title"] + " " + s.get("description", "")).lower()})

candidates = pd.DataFrame(rows).sort_values("comment_count", ascending=False).reset_index(drop=True)
candidates["found_by"] = candidates["video_id"].map(found)

KEYWORDS = ["reform", "curriculum", "grade 6", "grade 1", "module", "syllabus",
            "අධ්‍යාපන", "ප්‍රතිසංස්කරණ", "විෂය නිර්දේශ", "මොඩියුල", "ශ්‍රේණිය",
            "கல்வி", "சீர்திருத்த", "பாடத்திட்ட", "தரம்"]
MIN_COMMENTS = 20

candidates["auto_pass"] = (candidates["text"].apply(lambda t: any(k in t for k in KEYWORDS))
                           & (candidates["comment_count"] >= MIN_COMMENTS))

print(f"{len(candidates)} candidates -> {candidates['auto_pass'].sum()} pass the automatic filter")
pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_rows", 200)
candidates.loc[candidates["auto_pass"], ["video_id", "channel_title", "comment_count", "title"]]

377 candidates -> 52 pass the automatic filter


,video_id,channel_title,comment_count,title
0,RCj5wfAQWok,CNA Insider,2021,How Politics Is Behind Malaysia’s Education Reform Difficulties | Insight | ...
1,BpOaIFQOzJM,Maths කුප්පිය,1945,Grade 6 | වෘත්ත | Unit 01 | By @isharamadushanofficial
2,yln0Mbb--IQ,Ada Derana,1164,ලබන වසරේ සිට සා පෙළට අලුත් විෂයන් දෙකක් - අධ්‍යාපන ලේකම්ගෙන් පැහැදිලි කිරීමක...
3,-pHFRz2p-Nw,Maths with Chathura Madushan,730,2026 O/L Syllabus Change – Subject 7යි Credit 18යි | අලුත් විෂය ක්‍රමය මෙන්න
4,FbkrxtV5IKY,Newsfirst Sri Lanka,660,1 සිට 13 දක්වා අධ්‍යාපනයට වෙන්න යන වෙනස ගැන අගමැති හරිනි කළ පැහැදිලි කිරීම |...
5,R0S8yJajtuI,Ada Derana,589,2026 නව අධ්‍යාපන ප්‍රතිසංස්කරණ - පාසල් පැවැත්වෙන කාලසීමාවේත් වෙනසක්
6,WoD8P9g8OHs,BBC News සිංහල,552,නව අධ්‍යාපන ප්‍රතිසංස්කරණ සම්බන්ධයෙන් ජනාධිපති අනුර කුමාර දිසානායක පාර්ලිමේන...
7,Srr-sYiIDuI,BBC News සිංහල,512,නව අධ්‍යාපන ප්‍රතිසංස්කරණ සම්බන්ධයෙන් මාධ්‍යවේදීන් නැගූ ප්‍රශ්නවලට ලැබුණු පි...
8,QxHbridXiE0,Hiru News,265,6 වසර මොඩියුල වරද රජය භාරගනී - bමොඩියුලයේ ගැටලුව බරපතල වරදක් බවත් කියයි - Hi...
9,FKoS8yJOKWo,Edu Info,261,"R1 | Grade 6 Subject Selection Guide for Teachers, Students & Parents | Educ..."


**Output interpretation :** A threshold of 20 comments was chosen. A higher threshold (50) excluded relevant news and Tamil-language videos, and the 20–49 range added more variety in channels and languages.

fill in EXCLUDE_IDS after reading the titles

In [19]:
EXCLUDE_IDS = {
    # not about Sri Lanka
    "RCj5wfAQWok": "Malaysia education reform, not Sri Lanka",
    "5Gl_iWzY9UY": "SAARC countries in general, not Sri Lanka-specific",
    # subject lessons / class materials (comments about the lesson, not the reform)
    "BpOaIFQOzJM": "Grade 6 maths lesson",
    "fYqiOJ6cfdw": "advert for online maths classes",
    "AKWOP4GaTFM": "Grade 6 English class promotion",
    "uHp107rDADk": "Grade 6 module lesson (Day 1)",
    "SqajxXyuM-E": "Grade 6 ICT subject lesson",
    "xZ4WdYC_0qI": "Grade 6 science module lesson",
    "gC21s_wddnw": "Grade 6 maths module lesson",
    "fbCyhz0KSeA": "Grade 6 English module activities",
    "o9l47fQZ6xI": "Grade 6 maths module lesson",
    "BhQXKHDbNIs": "Grade 6 English syllabus PDF download",
    "Fbw7Hg7s738": "Grade 6 maths module lesson",
    # teacher exams / teacher career (comments about exams, not the reform)
    "ZYeuMI3SQzw": "teacher-exam general knowledge quiz",
    "ioOFm7cYB7M": "teacher-exam general knowledge quiz",
    "R9Onitso-Ic": "teacher-exam preparation",
    "mTKTus6XaWA": "education history for the teaching exam",
    "oVJKFQP9G6A": "graduate teacher exam preparation",
    "Q91XXZjywsc": "teacher promotion / EB exam system, not school reform",
}

candidates["selected"] = candidates["auto_pass"] & ~candidates["video_id"].isin(EXCLUDE_IDS)
candidates["exclude_reason"] = candidates["video_id"].map(EXCLUDE_IDS).fillna("")
candidates.drop(columns="text").to_csv(f"{DOCS_DIR}/candidate_videos.csv", index=False)

selected = candidates[candidates["selected"]].copy()
print(f"{len(selected)} videos | {selected['channel_title'].nunique()} channels | "
      f"~{selected['comment_count'].sum()} comments available")
selected[["video_id", "channel_title", "comment_count", "title"]]

33 videos | 21 channels | ~6454 comments available


,video_id,channel_title,comment_count,title
2,yln0Mbb--IQ,Ada Derana,1164,ලබන වසරේ සිට සා පෙළට අලුත් විෂයන් දෙකක් - අධ්‍යාපන ලේකම්ගෙන් පැහැදිලි කිරීමක...
3,-pHFRz2p-Nw,Maths with Chathura Madushan,730,2026 O/L Syllabus Change – Subject 7යි Credit 18යි | අලුත් විෂය ක්‍රමය මෙන්න
4,FbkrxtV5IKY,Newsfirst Sri Lanka,660,1 සිට 13 දක්වා අධ්‍යාපනයට වෙන්න යන වෙනස ගැන අගමැති හරිනි කළ පැහැදිලි කිරීම |...
5,R0S8yJajtuI,Ada Derana,589,2026 නව අධ්‍යාපන ප්‍රතිසංස්කරණ - පාසල් පැවැත්වෙන කාලසීමාවේත් වෙනසක්
6,WoD8P9g8OHs,BBC News සිංහල,552,නව අධ්‍යාපන ප්‍රතිසංස්කරණ සම්බන්ධයෙන් ජනාධිපති අනුර කුමාර දිසානායක පාර්ලිමේන...
7,Srr-sYiIDuI,BBC News සිංහල,512,නව අධ්‍යාපන ප්‍රතිසංස්කරණ සම්බන්ධයෙන් මාධ්‍යවේදීන් නැගූ ප්‍රශ්නවලට ලැබුණු පි...
8,QxHbridXiE0,Hiru News,265,6 වසර මොඩියුල වරද රජය භාරගනී - bමොඩියුලයේ ගැටලුව බරපතල වරදක් බවත් කියයි - Hi...
9,FKoS8yJOKWo,Edu Info,261,"R1 | Grade 6 Subject Selection Guide for Teachers, Students & Parents | Educ..."
12,UjmPunkM80A,Newsfirst Sri Lanka,164,🔴 නව අධ්‍යාපන ප්‍රතිසංස්කරණ - අගමැති හරිනි අමරසූරිය සමග සටන | Satana 16.09.2...
14,D02nj8GVQYA,Firstpost,152,Gay Website in School Syllabus Forces Sri Lanka to Delay Education Reforms |...


**1.4c: Tamil coverage**

In [20]:
has_kw = candidates["text"].apply(lambda t: any(k in t for k in KEYWORDS))
is_tamil = candidates["text"].str.contains(r"[\u0B80-\u0BFF]|tamil|தமிழ்")
tamil = candidates[is_tamil & has_kw]

print(f"Tamil-related reform videos found: {len(tamil)}")
print(f"  with >= {MIN_COMMENTS} comments: {(tamil['comment_count'] >= MIN_COMMENTS).sum()}")
print(f"  with 1-{MIN_COMMENTS - 1} comments: {tamil['comment_count'].between(1, MIN_COMMENTS - 1).sum()}")
print(f"  with 0 comments: {(tamil['comment_count'] == 0).sum()}")
print(f"Tamil-related videos in the final selection: {tamil['video_id'].isin(selected['video_id']).sum()}")

Tamil-related reform videos found: 89
  with >= 20 comments: 7
  with 1-19 comments: 30
  with 0 comments: 52
Tamil-related videos in the final selection: 7


**1.4d: topic coverage of the selected videos**

In [21]:
SCOPE_TOPICS = {
    "curriculum / subjects / modules": ["subject", "syllabus", "curriculum", "module", "විෂය", "මොඩියුල", "பாட"],
    "assessment / exams":              ["exam", "evaluation", "assessment", "gpa", "credit", "විභාග", "பரீட்சை", "மதிப்பீ"],
    "grade-level changes":             ["grade", "o/l", "a/l", "ශ්‍රේණිය", "වසර", "සා පෙළ", "සා.පෙළ", "උසස් පෙළ", "தரம்"],
    "school time / structure":         ["time table", "timetable", "කාලසීමා", "පැවැත්වෙන", "நேரம்", "மணி"],
    "policy / government":             ["parliament", "president", "minister", "cabinet", "අගමැති", "ජනාධිපති", "පාර්ලිමේන්තු", "பிரதமர்"],
    "problems / controversy":          ["error", "delay", "problem", "වරද", "ගැටලු", "பிரச்சினை"],
}
for topic, words in SCOPE_TOPICS.items():
    selected[topic] = selected["text"].apply(lambda t: any(w in t for w in words))

coverage = selected[list(SCOPE_TOPICS)].sum().sort_values(ascending=False).to_frame("videos")
coverage["% of selected"] = (100 * coverage["videos"] / len(selected)).round(0)
coverage

,videos,% of selected
curriculum / subjects / modules,18,55.0
grade-level changes,18,55.0
assessment / exams,12,36.0
policy / government,4,12.0
problems / controversy,4,12.0
school time / structure,2,6.0


**Topic coverage:** a keyword check on titles and descriptions (approximate) shows that the selected videos cover several reform areas listed in the brief's scope: [e.g. curriculum/modules (18 videos), assessment and exams (12), grade-level changes (18), school time (2), government policy statements (4)] and reported problems such as the Grade 6 module error (Hiru News) and the syllabus controversy that delayed the reform (Firstpost). No single video, channel or controversy dominates: the largest video holds ~[18]% of available comments, and the most frequent channel (Ada Derana) contributes [5] of 33 videos.

### 1.5 Comment collection
Comments are downloaded with `commentThreads.list`. Each video is saved to its own file in Google Drive so the download can resume after a disconnect.

**Data minimisation:** only the comment text and the metadata needed for analysis and reproducibility are kept: comment ID, parent ID, video and channel IDs, publication/update dates, like count and collection time.

**Replies:** replies are kept and marked with `is_reply` and `parent_id`. The API returns at most 5 replies per thread, so longer reply chains are only partly collected.

**Unavailable comments:** if a video's comments are disabled or the video is no longer available, the error is logged in `search_log.csv` and the video is skipped.

**1.5a: download**

In [22]:
def make_row(comment_id, parent_id, s, video, collected_at):
    return {
        "comment_id": comment_id,
        "parent_id": parent_id,
        "is_reply": parent_id is not None,
        "video_id": video["video_id"],
        "channel_id": video["channel_id"],
        "channel_title": video["channel_title"],
        "text": s.get("textDisplay", ""),
        "like_count": s.get("likeCount", 0),
        "published_at": s.get("publishedAt"),
        "updated_at": s.get("updatedAt"),
        "collected_at": collected_at,
    }

def get_comments(video):
    rows, page_token, collected_at = [], None, now_utc()
    while True:
        r = api_call(yt.commentThreads().list(part="snippet,replies",
                                              videoId=video["video_id"],
                                              maxResults=100, pageToken=page_token,
                                              textFormat="plainText"),
                     "commentThreads", video["video_id"], 1)
        if r is None:                        # comments disabled / video unavailable
            break
        for th in r["items"]:
            top = th["snippet"]["topLevelComment"]
            rows.append(make_row(top["id"], None, top["snippet"], video, collected_at))
            for rep in th.get("replies", {}).get("comments", []):
                rows.append(make_row(rep["id"], top["id"], rep["snippet"], video, collected_at))
        page_token = r.get("nextPageToken")  # another page?
        if not page_token:
            break
    return rows

for _, video in selected.iterrows():
    path = f"{RAW_DIR}/{video['video_id']}.jsonl"
    if os.path.exists(path):                 # resume: skip finished videos
        print(f"already downloaded: {video['video_id']}")
        continue
    rows = get_comments(video)
    pd.DataFrame(rows).to_json(path, orient="records", lines=True, force_ascii=False)
    print(f"{video['video_id']} | {video['channel_title'][:25]:25} | {len(rows):5} comments")

print("Quota used this session:", QUOTA_USED)

yln0Mbb--IQ | Ada Derana                |  1065 comments
-pHFRz2p-Nw | Maths with Chathura Madus |   672 comments
FbkrxtV5IKY | Newsfirst Sri Lanka       |   637 comments
R0S8yJajtuI | Ada Derana                |   551 comments
WoD8P9g8OHs | BBC News සිංහල            |   539 comments
Srr-sYiIDuI | BBC News සිංහල            |   504 comments
QxHbridXiE0 | Hiru News                 |   242 comments
FKoS8yJOKWo | Edu Info                  |   261 comments
UjmPunkM80A | Newsfirst Sri Lanka       |   163 comments
D02nj8GVQYA | Firstpost                 |   116 comments
wyzYk_NyOCQ | VFM RADIO 107             |   145 comments
4A3C19_JGHU | Ashani Science LAB        |   104 comments
SlRJNlPAj8A | Manoj Dhananjaya          |   101 comments
6UIYSBCgP2E | Ada Derana                |    89 comments
TZSJhNEVtS8 | ARV Loshan News           |    93 comments
9N2ueXbq4vg | Bio Api                   |    89 comments
TiEcV1uToC8 | Ada Derana                |    76 comments
bxc2J8bflxk | Nirmala Jayasoori

**1.5b: download summary**

In [23]:
def n_lines(path):
    if not os.path.exists(path):
        return 0
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

selected["n_downloaded"] = selected["video_id"].apply(lambda v: n_lines(f"{RAW_DIR}/{v}.jsonl"))

total_dl, total_yt = selected["n_downloaded"].sum(), selected["comment_count"].sum()
print(f"Downloaded {total_dl} of {total_yt} comments shown by YouTube ({total_dl / total_yt:.0%})")
print("Videos with nothing downloaded:", (selected["n_downloaded"] == 0).sum())

selected[["video_id", "channel_title", "comment_count", "n_downloaded"]]

Downloaded 6133 of 6454 comments shown by YouTube (95%)
Videos with nothing downloaded: 0


,video_id,channel_title,comment_count,n_downloaded
2,yln0Mbb--IQ,Ada Derana,1164,1065
3,-pHFRz2p-Nw,Maths with Chathura Madushan,730,672
4,FbkrxtV5IKY,Newsfirst Sri Lanka,660,637
5,R0S8yJajtuI,Ada Derana,589,551
6,WoD8P9g8OHs,BBC News සිංහල,552,539
7,Srr-sYiIDuI,BBC News සිංහල,512,504
8,QxHbridXiE0,Hiru News,265,242
9,FKoS8yJOKWo,Edu Info,261,261
12,UjmPunkM80A,Newsfirst Sri Lanka,164,163
14,D02nj8GVQYA,Firstpost,152,116


**Output interpretation:** Comments were collected on 15 Nov 2026 (UTC). **6,133 comments** were downloaded, 95% of the 6,454 shown by YouTube. No video was skipped, so no selected video had comments disabled.

Most smaller videos were collected completely (100%). The missing 5% is concentrated in large or controversial videos: Firstpost (116 of 152, 24% missing), Hiru News (242 of 265) and Ada Derana (1,065 of 1,164). This is explained by:
1. the API returning at most 5 replies per thread, which affects videos with long discussions most; and
2. deleted, hidden or held-for-review comments not being returned by the API.

Downloads range from 1,065 comments (Ada Derana, yln0Mbb--IQ) to 20 (one Ada Derana and one Newsfirst video).

### 1.6 Corpus assembly and requirement checks
All per-video files are combined into one corpus. Duplicate comment IDs and empty comments are removed, the 30% per-video rule is checked (over-represented videos would be randomly down-sampled with seed 42), and the corpus is checked against the Task 1 requirements.

**1.6a: combine and clean**

In [25]:
ID_TYPES = {"comment_id": str, "parent_id": str, "video_id": str, "channel_id": str}

frames = []
for vid in selected["video_id"]:
    path = f"{RAW_DIR}/{vid}.jsonl"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        frames.append(pd.read_json(path, lines=True, convert_dates=False, dtype=ID_TYPES))

raw = pd.concat(frames, ignore_index=True)
n_raw = len(raw)

n_dup_id = raw["comment_id"].duplicated().sum()
raw = raw.drop_duplicates("comment_id")

raw["text"] = raw["text"].fillna("").str.strip()
n_empty = (raw["text"] == "").sum()
raw = raw[raw["text"] != ""].reset_index(drop=True)

print(f"Downloaded rows: {n_raw}")
print(f"Duplicate comment IDs removed: {n_dup_id}")
print(f"Empty comments removed: {n_empty}")
print(f"Remaining: {len(raw)}")
raw.to_json(f"{PROC_DIR}/comments_all_raw.jsonl", orient="records", lines=True, force_ascii=False)

Downloaded rows: 6133
Duplicate comment IDs removed: 6
Empty comments removed: 0
Remaining: 6127


**Output interpretation:** The 33 files contained 6,133 rows. 6 duplicate comment IDs were removed; these occur when new comments arrive during paging and a comment is returned on two pages. No comments had empty text. 6,127 unique comments remain.

**1.6b: 30% rule**

In [26]:
def apply_video_cap(df, max_share=0.30, seed=SEED):
    """Randomly down-sample the largest video until no video exceeds max_share."""
    df = df.copy()
    while True:
        counts = df["video_id"].value_counts()
        top_vid, top_n = counts.index[0], counts.iloc[0]
        if top_n / counts.sum() <= max_share:
            return df.reset_index(drop=True)
        others = counts.sum() - top_n
        allowed = int(max_share * others / (1 - max_share))
        keep_idx = df[df["video_id"] == top_vid].sample(n=allowed, random_state=seed).index
        df = df[(df["video_id"] != top_vid) | df.index.isin(keep_idx)]

corpus = apply_video_cap(raw)
print(f"{len(raw)} -> {len(corpus)} comments after the 30% check "
      f"({'no down-sampling needed' if len(corpus) == len(raw) else 'down-sampling applied'})")

6127 -> 6127 comments after the 30% check (no down-sampling needed)


**Output interpretation:** No down-sampling was needed. The largest video (Ada Derana, yln0Mbb--IQ) contributes 17.4% of the corpus, below the 30% limit, so no exception had to be justified.

**1.6c: requirement checks and save**

In [27]:
share = corpus["video_id"].value_counts(normalize=True)
checks = {
    "at least 3,000 unique comments": corpus["comment_id"].nunique() >= 3000,
    "at least 3 distinct channels":   corpus["channel_id"].nunique() >= 3,
    "no video above 30% of comments": share.max() <= 0.30,
}
for name, ok in checks.items():
    print("True" if ok else "False", name)

print(f"\n{len(corpus)} comments | {corpus['channel_id'].nunique()} channels | "
      f"{corpus['video_id'].nunique()} videos | largest video share: {share.max():.1%}")
print(f"Top-level: {(~corpus['is_reply']).sum()} | replies: {corpus['is_reply'].sum()}")
print(f"Unique comment texts: {corpus['text'].nunique()} "
      f"({len(corpus) - corpus['text'].nunique()} comments repeat an earlier text)")

corpus.to_json(f"{PROC_DIR}/comments_task1.jsonl", orient="records", lines=True, force_ascii=False)

video_list = selected[["video_id", "channel_id", "channel_title", "title",
                       "published_at", "comment_count", "found_by"]].copy()
video_list["n_in_corpus"] = video_list["video_id"].map(corpus["video_id"].value_counts()).fillna(0).astype(int)
video_list.to_csv(f"{DOCS_DIR}/video_list.csv", index=False)
print("\nSaved comments_task1.jsonl and video_list.csv")

True at least 3,000 unique comments
True at least 3 distinct channels
True no video above 30% of comments

6127 comments | 21 channels | 33 videos | largest video share: 17.4%
Top-level: 4442 | replies: 1685
Unique comment texts: 5613 (514 comments repeat an earlier text)

Saved comments_task1.jsonl and video_list.csv


**Output interpretation:** The final corpus meets all Task 1 requirements: **6,127 unique comments** (more than double the 3,000 minimum) from **33 videos across 21 channels**, with no video above 17.4%. 4,442 are top-level comments and 1,685 (27.5%) are replies, so a substantial part of the corpus is conversation between commenters. 514 comments (8.4%) repeat the exact text of an earlier comment; these are kept for the EDA and handled in Task 2 preprocessing.

**How data issues were handled (Task 1b):**
- **Comments disabled / unavailable:** videos with 0 comments were excluded at screening (1.4). All 33 selected videos were downloaded successfully; any API errors would have been logged and the video skipped.
- **Duplicates:** 6 duplicate comment IDs were removed. Comments with identical text but different IDs (514) are measured in the EDA and handled in Task 2.
- **Replies:** kept (1,685) and marked with `is_reply` and `parent_id`. The API returns at most 5 replies per thread, so long reply chains are only partly collected.
- **Deleted comments:** the API returns only comments that exist at collection time, so deleted or hidden comments are absent. This is part of why 6,133 rather than 6,454 comments were downloaded (1.5).
- **Spam:** YouTube holds back likely spam before it reaches the API. Remaining repeated, link-only or emoji-only comments are measured in 1.7 and removed in Task 2.
- **API limitations:** daily quota of 10,000 units (the whole collection used about 1495 units); at most 50 results per search page; search ranking changes over time (26 Oct vs 15 Nov test runs). The search log, video list and collection timestamps make the corpus traceable even though a re-run would differ.